# BCS407 v2 Training on Google Colab

This notebook rebuilds the dataset, optionally augments it, trains YOLOv8m, and saves checkpoints to Google Drive so you can resume later.

## 1. Runtime

In Colab, set `Runtime > Change runtime type > T4 GPU` or any available GPU before running the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/MohammadThabetHassan/bcs407-campus-safety.git'
REPO_DIR = Path('/content/bcs407-campus-safety')
DRIVE_ROOT = Path('/content/drive/MyDrive/bcs407-campus-safety')
ZIP_DIR = DRIVE_ROOT / 'zips'
RUNS_BACKUP_DIR = DRIVE_ROOT / 'runs'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)
RUNS_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print('Put these zip files in:', ZIP_DIR)
print('- wet-floor-detection1.v2i.yolov8.zip')
print('- Fire Alarm.v24i.yolov8 (1).zip')
print('- Emergency Exit Signs.v4i.yolov8.zip')
print('- Hard Hat Universe.v4i.yolov8.zip')

In [ ]:
!rm -rf /content/bcs407-campus-safety
!git clone $REPO_URL /content/bcs407-campus-safety
%cd /content/bcs407-campus-safety
!python -m pip install --upgrade pip
!pip install -r requirements.txt

In [ ]:
required = [
    'wet-floor-detection1.v2i.yolov8.zip',
    'Fire Alarm.v24i.yolov8 (1).zip',
    'Emergency Exit Signs.v4i.yolov8.zip',
    'Hard Hat Universe.v4i.yolov8.zip',
]
missing = [name for name in required if not (ZIP_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing zip files in {ZIP_DIR}: {missing}')

for name in required:
    src = ZIP_DIR / name
    dst = REPO_DIR / name
    if not dst.exists():
        dst.write_bytes(src.read_bytes())

print('Zip files copied into repo root.')

In [ ]:
%cd /content/bcs407-campus-safety
!python code/setup_v2.py
!python code/augment_v2.py

## 2. Train

Default config uses `batch=32` and `workers=0`, which is safer for Colab shared-memory limits.

In [ ]:
%cd /content/bcs407-campus-safety
!bash code/train_v2.sh

## 3. Save checkpoints to Drive

Run this after training, or any time during training after weights are created.

In [ ]:
%cd /content/bcs407-campus-safety
!python code/backup_run_artifacts.py --dest "$RUNS_BACKUP_DIR/campus_safety_v2_fixed"

## 4. Resume later

If Colab disconnects, restore the backed-up artifacts from Drive and resume from `last.pt`.

In [ ]:
%cd /content/bcs407-campus-safety
!mkdir -p runs/detect/campus_safety_v2_fixed/weights
!cp "$RUNS_BACKUP_DIR/campus_safety_v2_fixed/weights/last.pt" runs/detect/campus_safety_v2_fixed/weights/last.pt
!yolo detect train resume model=runs/detect/campus_safety_v2_fixed/weights/last.pt